In [2]:
SCALE_BITS = 10
SCALE = 1 << SCALE_BITS

def normalize_counts(order, counts, total, scale=SCALE):
    freqs = []
    for v in order:
        c = counts.get(v, 0)
        f = int(c / total * scale)
        if f == 0:
            f = 1
        freqs.append(f)

    diff = scale - sum(freqs)
    i_max = max(range(len(freqs)), key=lambda i: freqs[i])
    freqs[i_max] += diff

    return freqs, diff, i_max

# 128 bits 安全性的 rANS 编码预计算

In [3]:
# h 样本个数：51200000
total_h = 51_200_000
counts_h = {
    -1023: 11230, -1022: 9232, -1021: 2963, -1020: 463, -1019: 26,
    -6: 2, -5: 6652, -4: 118072, -3: 1005869, -2: 4656777, -1: 11744521,
    0: 16095334, 1: 11734295, 2: 4659871, 3: 1006660, 4: 117735, 5: 6397, 6: 1,
    1019: 42, 1020: 449, 1021: 2936, 1022: 9113, 1023: 11360
}

# 映射关系: -6..6 -> 0..12, -1023..-1019 -> 13..17, 1019..1023 -> 18..22
order_h = list(range(-6, 7)) + list(range(-1023, -1018)) + list(range(1019, 1024))

# hb(z1) 样本个数：102400000
total_hb = 102_400_000
counts_hb = {
    0: 32916405, 1: 23789372, 2: 8978273, 3: 1784492, 4: 184933, 5: 8327,
    1019: 8527, 1020: 186534, 1021: 1787677, 1022: 8974499, 1023: 23780961
}

# 映射关系: 0..5 -> 0..5, 1019..1023 -> 6..10
order_hb = [0, 1, 2, 3, 4, 5, 1019, 1020, 1021, 1022, 1023]

f_h, diff_h, i_max_h = normalize_counts(order_h, counts_h, total_h)
f_hb_z1, diff_hb, i_max_hb = normalize_counts(order_hb, counts_hb, total_hb)


def c_array(name, arr):
    return f'static const uint32_t {name}[{len(arr)}] = {{' + ', '.join(map(str, arr)) + '};'

print('#define M_H', len(f_h))
print(c_array('f_h', f_h))
print()
print('#define M_HB_Z1', len(f_hb_z1))
print(c_array('f_hb_z1', f_hb_z1))

#define M_H 23
static const uint32_t f_h[23] = {1, 1, 2, 20, 93, 234, 312, 234, 93, 20, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1};

#define M_HB_Z1 11
static const uint32_t f_hb_z1[11] = {334, 237, 89, 17, 1, 1, 1, 1, 17, 89, 237};


# 256 bits 安全性的 rANS 编码预计算

In [15]:
# h 样本个数：102400000
total_h_256 = 102_400_000
counts_h_256 = {
    -9: 1455, -8: 18706, -7: 96066, -6: 387375, -5: 1263813,
    -4: 3333087, -3: 7090450, -2: 12227834, -1: 17043980,
    0: 19152807,
    1: 17040279, 2: 12217935, 3: 7092198, 4: 3328570, 5: 1266243,
    6: 386968, 7: 95940, 8: 18710, 9: 1576,
    -511: 33663, -510: 48255, -509: 41646, -508: 26084, -507: 12489,
    -506: 4446, -505: 1331, -504: 315, -503: 38,
    503: 28, 504: 273, 505: 1337, 506: 4580, 507: 12286,
    508: 26085, 509: 41662, 510: 47822, 511: 33668
}

# 映射关系: -9..9 -> 0..18, -511..-503 -> 19..27, 503..511 -> 28..36
order_h_256 = list(range(-9, 10)) + list(range(-511, -502)) + list(range(503, 512))

# hb(z1) 样本个数：153600000
total_hb_256 = 153_600_000
counts_hb_256 = {
    0: 29271205, 1: 25748170, 2: 18374426, 3: 10569061, 4: 4903444,
    5: 1828871, 6: 550654, 7: 132054, 8: 25638, 9: 1346,
    503: 1267, 504: 25493, 505: 133896, 506: 549983, 507: 1831843,
    508: 4903888, 509: 10585540, 510: 18397353, 511: 25765868
}

# 映射关系: 0..9 -> 0..9, 503..511 -> 10..18
order_hb_256 = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 503, 504, 505, 506, 507, 508, 509, 510, 511]

f_h_256, diff_h_256, i_max_h_256 = normalize_counts(order_h_256, counts_h_256, total_h_256)
f_hb_z1_256, diff_hb_256, i_max_hb_256 = normalize_counts(order_hb_256, counts_hb_256, total_hb_256)

print(f"#define M_H {len(f_h_256)}")
print(f"static const uint32_t f_h[{len(f_h_256)}] = {{" + ", ".join(map(str, f_h_256)) + "};")
print()
print(f"#define M_HB_Z1 {len(f_hb_z1_256)}")
print(f"static const uint32_t f_hb_z1[{len(f_hb_z1_256)}] = {{" + ", ".join(map(str, f_hb_z1_256)) + "};")
print()
print(f"Sum f_h = {sum(f_h_256)}, Sum f_hb_z1 = {sum(f_hb_z1_256)}")

#define M_H 37
static const uint32_t f_h[37] = {1, 1, 1, 3, 12, 33, 70, 122, 170, 180, 170, 122, 70, 33, 12, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1};

#define M_HB_Z1 19
static const uint32_t f_hb_z1[19] = {198, 171, 122, 70, 32, 12, 3, 1, 1, 1, 1, 1, 1, 3, 12, 32, 70, 122, 171};

Sum f_h = 1024, Sum f_hb_z1 = 1024
